In [4]:
import pandas as pd

# Load the data (adjust path if needed)
file_path = "/home/emchatwin/Kelly Lab/ms_qc_processor/input2/bachelorette.csv"
df = pd.read_csv(file_path, skiprows=[1])   # skip the second header row

# Define column groups
elim_cols = [f'ELIM_{i}' for i in range(1, 11)]
date_cols = [f'DATE_{i}' for i in range(1, 11)]
df.columns = ['SHOW', 'SEASON', 'CONTESTANT'] + elim_cols + date_cols

# Remove separator rows (where SEASON == 'SEASON')
df = df[df['SEASON'] != 'SEASON'].copy()

# Elimination codes that mean "eliminated"
elim_codes = {'E', 'ED', 'EU', 'EQ', 'EF'}

def get_elim_week(row):
    for w in range(1, 11):
        val = row[f'ELIM_{w}']
        if val in elim_codes:
            return w
        if val == 'W':
            return 11   # winner – survives all weeks
    return 11

df['ELIM_WEEK'] = df.apply(get_elim_week, axis=1)

# Build weekly timeline
records = []
for _, row in df.iterrows():
    show = row['SHOW']
    season = row['SEASON']
    contestant = row['CONTESTANT']
    elim_week = row['ELIM_WEEK']
    for week in range(1, 11):
        alive = week < elim_week
        date_val = row[f'DATE_{week}']
        one_on_one = (date_val == 'D1')
        records.append({
            'SHOW': show,
            'SEASON': season,
            'CONTESTANT': contestant,
            'WEEK': week,
            'ALIVE': alive,
            'ONE_ON_ONE': one_on_one,
            'ELIMINATED_THIS_WEEK': (week == elim_week and elim_week <= 10)
        })

weekly_df = pd.DataFrame(records)

# Weekly ratio of one‑on‑one dates among alive contestants
ratio_df = (weekly_df[weekly_df['ALIVE']]
            .groupby(['SHOW', 'SEASON', 'WEEK'])
            .apply(lambda g: pd.Series({
                'TOTAL_ALIVE': len(g),
                'NUM_ONE_ON_ONE': g['ONE_ON_ONE'].sum(),
                'RATIO_ONE_ON_ONE': g['ONE_ON_ONE'].sum() / len(g)
            }))
            .reset_index())

# Save cleaned data
weekly_df.to_csv('contestant_weekly.csv', index=False)
ratio_df.to_csv('weekly_ratio.csv', index=False)

print("Files saved: contestant_weekly.csv and weekly_ratio.csv")

Files saved: contestant_weekly.csv and weekly_ratio.csv


/tmp/ipykernel_520781/3240000354.py:55: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
